# Django, Forms

## Introduction to Django Forms
---

In this lesson, you'll learn how to **handle user input with Django Forms**.

Forms are essential for any web application that needs to collect data from users - whether it's a login page, a contact form, or a product order.

1. [Why Use Django Forms](#Why-Use-Django-Forms),
    - [The Problem with Raw HTML Forms](#The-Problem-with-Raw-HTML-Forms),
    - [What Django Forms Provide](#What-Django-Forms-Provide),
2. [Creating Your First Form](#Creating-Your-First-Form),
    - [Form Class and Fields](#Form-Class-and-Fields),
    - [Common Field Types](#Common-Field-Types),
3. [Rendering Forms in Templates](#Rendering-Forms-in-Templates),
    - [Quick Rendering Methods](#Quick-Rendering-Methods),
    - [Manual Field Rendering](#Manual-Field-Rendering),
    - [CSRF Protection](#CSRF-Protection),
4. [Processing Form Data](#Processing-Form-Data),
    - [The View Pattern](#The-View-Pattern),
    - [is_valid() and cleaned_data](#is_valid()-and-cleaned_data),
5. [Basic Field Validation](#Basic-Field-Validation),
    - [Built-in Validators](#Built-in-Validators),
    - [Field Arguments](#Field-Arguments),
6. [🧠 EXERCISE 🧠, Create a Contact Form](#🧠-EXERCISE-🧠,-Create-a-Contact-Form),
7. [🧠 EXERCISE 🧠, Build a Newsletter Signup](#🧠-EXERCISE-🧠,-Build-a-Newsletter-Signup).

<br>

## Why Use Django Forms

---

### The Problem with Raw HTML Forms

---

You could handle forms with plain HTML and manually process `request.POST`:

```html
<form method="post">
    <input type="text" name="email">
    <input type="submit" value="Subscribe">
</form>
```

<br>

But this approach has **serious problems**:

| Problem | Risk |
|---------|------|
| No automatic validation | Invalid data gets into your database |
| No type conversion | Everything is a string |
| Manual error handling | Inconsistent user experience |
| Security vulnerabilities | XSS, CSRF attacks |
| Repetitive code | You write the same logic repeatedly |

<br>

### What Django Forms Provide

---

Django Forms solve all of these problems:

<br>

| Feature | Benefit |
|---------|--------|
| **Automatic HTML generation** | Consistent, accessible markup |
| **Data validation** | Invalid data is rejected before processing |
| **Type conversion** | Strings become proper Python types |
| **Error handling** | User-friendly error messages |
| **CSRF protection** | Built-in security against attacks |
| **Reusability** | Define once, use everywhere |

<br>

## Creating Your First Form

---

### Form Class and Fields

---

Django forms are defined as Python classes that inherit from `forms.Form`.

Create a new file `forms.py` in your app directory:

In [ ]:
# catalog/forms.py

from django import forms


class ContactForm(forms.Form):
    """A simple contact form for user inquiries."""
    
    name = forms.CharField(max_length=100)
    email = forms.EmailField()
    message = forms.CharField(widget=forms.Textarea)

<br>

**Key concepts:**

- Each **field** corresponds to an input element in the HTML form
- Field types (like `CharField`, `EmailField`) determine validation and HTML widget
- The **widget** controls how the field appears (text input, textarea, checkbox, etc.)

<br>

### Common Field Types

---

Django provides many built-in field types for different data:

<br>

| Field Type | HTML Widget | Python Type | Use For |
|------------|------------|-------------|--------|
| `CharField` | `<input type="text">` | `str` | Short text |
| `EmailField` | `<input type="email">` | `str` | Email addresses |
| `IntegerField` | `<input type="number">` | `int` | Whole numbers |
| `DecimalField` | `<input type="number">` | `Decimal` | Money, precise numbers |
| `BooleanField` | `<input type="checkbox">` | `bool` | Yes/No choices |
| `DateField` | `<input type="text">` | `date` | Dates |
| `ChoiceField` | `<select>` | `str` | Dropdown selection |
| `FileField` | `<input type="file">` | `UploadedFile` | File uploads |

In [ ]:
# Example: A more complex form with various field types

from django import forms


class BookOrderForm(forms.Form):
    """Form for ordering books from the catalog."""
    
    DELIVERY_CHOICES = [
        ('standard', 'Standard Delivery (5-7 days)'),
        ('express', 'Express Delivery (2-3 days)'),
        ('overnight', 'Overnight Delivery'),
    ]
    
    book_title = forms.CharField(max_length=200)
    quantity = forms.IntegerField(min_value=1, max_value=100)
    email = forms.EmailField()
    delivery = forms.ChoiceField(choices=DELIVERY_CHOICES)
    gift_wrap = forms.BooleanField(required=False)  # Optional checkbox
    notes = forms.CharField(
        widget=forms.Textarea,
        required=False  # Optional field
    )

<br>

## Rendering Forms in Templates

---

### Quick Rendering Methods

---

Django provides several shortcuts to render the entire form at once:

<br>

| Method | Output Format |
|--------|---------------|
| `{{ form.as_p }}` | Each field wrapped in `<p>` tags |
| `{{ form.as_table }}` | Each field in a `<tr>` (requires `<table>`) |
| `{{ form.as_ul }}` | Each field in an `<li>` (requires `<ul>`) |
| `{{ form.as_div }}` | Each field wrapped in `<div>` tags (Django 4.0+) |

In [ ]:
# Example template: templates/catalog/contact.html

template_example = """
<!DOCTYPE html>
<html>
<head>
    <title>Contact Us</title>
</head>
<body>
    <h1>Contact Us</h1>
    
    <form method="post">
        {% csrf_token %}
        {{ form.as_p }}
        <button type="submit">Send Message</button>
    </form>
</body>
</html>
"""
print(template_example)

<br>

**Important:** Always include `{% csrf_token %}` inside your form! This is required for security.

<br>

### Manual Field Rendering

---

For more control over the HTML output, you can render fields individually:

In [ ]:
# Manual field rendering example

template_manual = """
<form method="post">
    {% csrf_token %}
    
    <div class="form-group">
        <label for="{{ form.name.id_for_label }}">Your Name:</label>
        {{ form.name }}
        {% if form.name.errors %}
            <span class="error">{{ form.name.errors.0 }}</span>
        {% endif %}
    </div>
    
    <div class="form-group">
        <label for="{{ form.email.id_for_label }}">Email:</label>
        {{ form.email }}
        {% if form.email.errors %}
            <span class="error">{{ form.email.errors.0 }}</span>
        {% endif %}
    </div>
    
    <div class="form-group">
        <label for="{{ form.message.id_for_label }}">Message:</label>
        {{ form.message }}
    </div>
    
    <button type="submit">Send</button>
</form>
"""
print(template_manual)

<br>

**Available attributes for each field:**

| Attribute | Description |
|-----------|-------------|
| `{{ form.fieldname }}` | The input widget itself |
| `{{ form.fieldname.label }}` | The field's label text |
| `{{ form.fieldname.id_for_label }}` | The HTML id attribute |
| `{{ form.fieldname.errors }}` | List of validation errors |
| `{{ form.fieldname.help_text }}` | Help text if defined |
| `{{ form.fieldname.value }}` | The current value |

<br>

### CSRF Protection

---

**CSRF** (Cross-Site Request Forgery) is an attack where a malicious site tricks a user's browser into submitting a form to your site.

<br>

Django protects against this with a token:

1. When rendering the form, Django generates a unique token
2. The token is included as a hidden field in the form
3. When the form is submitted, Django verifies the token matches

<br>

**Always include the CSRF token in POST forms:**

```html
<form method="post">
    {% csrf_token %}  <!-- This is REQUIRED -->
    {{ form.as_p }}
    <button type="submit">Submit</button>
</form>
```

Without `{% csrf_token %}`, Django will reject the form submission with a 403 Forbidden error.

<br>

## Processing Form Data

---

### The View Pattern

---

The standard pattern for handling forms in Django views:

In [ ]:
# catalog/views.py

from django.shortcuts import render, redirect
from .forms import ContactForm


def contact(request):
    """Handle the contact form - display and process."""
    
    if request.method == 'POST':
        # Form was submitted - bind data to form
        form = ContactForm(request.POST)
        
        if form.is_valid():
            # Data is valid - process it
            name = form.cleaned_data['name']
            email = form.cleaned_data['email']
            message = form.cleaned_data['message']
            
            # Do something with the data (e.g., send email)
            print(f"Message from {name} ({email}): {message}")
            
            # Redirect to success page (PRG pattern)
            return redirect('contact_success')
    else:
        # GET request - show empty form
        form = ContactForm()
    
    return render(request, 'catalog/contact.html', {'form': form})

<br>

**The flow explained:**

1. **GET request**: User visits the page → show empty form
2. **POST request**: User submits the form → validate and process
3. **Valid data**: Process data and redirect (PRG pattern)
4. **Invalid data**: Show form again with errors

<br>

**PRG Pattern** (Post-Redirect-Get): After successful form submission, always redirect. This prevents duplicate submissions when the user refreshes the page.

<br>

### is_valid() and cleaned_data

---

Two key concepts when processing form data:

<br>

**`is_valid()`** - Validates all form fields:
- Returns `True` if all fields pass validation
- Returns `False` if any field has errors
- Populates `form.errors` with error messages

In [ ]:
# Example: Checking form validity

from django import forms


class SimpleForm(forms.Form):
    email = forms.EmailField()
    age = forms.IntegerField(min_value=0)


# Simulate form with invalid data
form = SimpleForm(data={'email': 'not-an-email', 'age': '-5'})

print(f"Is valid: {form.is_valid()}")
print(f"Errors: {form.errors}")

<br>

**`cleaned_data`** - Dictionary of validated and converted data:
- Only available after calling `is_valid()` and getting `True`
- Contains Python objects, not raw strings
- Missing optional fields have their default values

In [ ]:
# Example: Accessing cleaned_data

form = SimpleForm(data={'email': 'user@example.com', 'age': '25'})

if form.is_valid():
    print(f"Email: {form.cleaned_data['email']}")
    print(f"Age: {form.cleaned_data['age']}")
    print(f"Age type: {type(form.cleaned_data['age'])}")

<br>

**Important:** Always access user data through `cleaned_data`, never directly from `request.POST`. The `cleaned_data` values are:
- Validated
- Converted to proper Python types
- Sanitized

<br>

## Basic Field Validation

---

### Built-in Validators

---

Django provides many built-in validators that you can attach to fields:

In [ ]:
from django import forms
from django.core.validators import (
    MinLengthValidator,
    MaxLengthValidator,
    MinValueValidator,
    MaxValueValidator,
    RegexValidator,
    EmailValidator,
    URLValidator,
)


class RegistrationForm(forms.Form):
    """User registration form with various validators."""
    
    username = forms.CharField(
        min_length=3,
        max_length=30,
        validators=[
            RegexValidator(
                regex=r'^[a-zA-Z0-9_]+$',
                message='Username can only contain letters, numbers, and underscores.'
            )
        ]
    )
    
    email = forms.EmailField()  # Has built-in email validation
    
    age = forms.IntegerField(
        validators=[
            MinValueValidator(18, message='You must be at least 18 years old.'),
            MaxValueValidator(120, message='Please enter a valid age.')
        ]
    )
    
    website = forms.URLField(required=False)  # Has built-in URL validation

<br>

| Validator | Purpose | Example |
|-----------|---------|--------|
| `MinLengthValidator` | Minimum string length | `MinLengthValidator(3)` |
| `MaxLengthValidator` | Maximum string length | `MaxLengthValidator(100)` |
| `MinValueValidator` | Minimum numeric value | `MinValueValidator(0)` |
| `MaxValueValidator` | Maximum numeric value | `MaxValueValidator(100)` |
| `RegexValidator` | Match a pattern | `RegexValidator(r'^[A-Z]+$')` |
| `EmailValidator` | Valid email (automatic in EmailField) | Built into EmailField |
| `URLValidator` | Valid URL (automatic in URLField) | Built into URLField |

<br>

### Field Arguments

---

Every field accepts common arguments for validation and display:

In [ ]:
from django import forms


class DetailedForm(forms.Form):
    """Form demonstrating common field arguments."""
    
    # required (default=True) - field must have a value
    required_field = forms.CharField(required=True)  # Default
    optional_field = forms.CharField(required=False)
    
    # initial - pre-filled value when form is displayed
    country = forms.CharField(initial='United States')
    
    # label - custom label text (default is field name capitalized)
    email_address = forms.EmailField(label='Your Email')
    
    # help_text - additional guidance for the user
    password = forms.CharField(
        widget=forms.PasswordInput,
        help_text='Must be at least 8 characters.'
    )
    
    # error_messages - customize error messages
    age = forms.IntegerField(
        min_value=0,
        error_messages={
            'required': 'Please enter your age.',
            'min_value': 'Age cannot be negative.',
            'invalid': 'Please enter a valid number.'
        }
    )

<br>

| Argument | Purpose | Example |
|----------|---------|--------|
| `required` | Whether field must have a value | `required=False` |
| `initial` | Default value for display | `initial='US'` |
| `label` | Custom label text | `label='Your Name'` |
| `help_text` | Guidance text | `help_text='Enter your email'` |
| `widget` | HTML input type | `widget=forms.Textarea` |
| `error_messages` | Custom error messages | `{'required': 'This is required'}` |
| `validators` | List of validator functions | `validators=[my_validator]` |

<br>

## Complete Example: Contact Form

---

Let's put it all together with a complete working example:

In [ ]:
# catalog/forms.py

from django import forms


class ContactForm(forms.Form):
    """Contact form for customer inquiries."""
    
    SUBJECT_CHOICES = [
        ('', '-- Select Subject --'),
        ('general', 'General Inquiry'),
        ('support', 'Technical Support'),
        ('billing', 'Billing Question'),
        ('feedback', 'Feedback'),
    ]
    
    name = forms.CharField(
        max_length=100,
        label='Your Name',
        widget=forms.TextInput(attrs={'placeholder': 'John Doe'})
    )
    
    email = forms.EmailField(
        label='Email Address',
        widget=forms.EmailInput(attrs={'placeholder': 'john@example.com'})
    )
    
    subject = forms.ChoiceField(
        choices=SUBJECT_CHOICES,
        label='Subject'
    )
    
    message = forms.CharField(
        widget=forms.Textarea(attrs={'rows': 5, 'placeholder': 'Your message...'}),
        label='Message',
        min_length=10,
        help_text='Please provide at least 10 characters.'
    )
    
    copy_to_self = forms.BooleanField(
        required=False,
        label='Send me a copy',
        initial=True
    )

In [ ]:
# catalog/views.py

from django.shortcuts import render, redirect
from django.contrib import messages
from .forms import ContactForm


def contact(request):
    """Display and process the contact form."""
    
    if request.method == 'POST':
        form = ContactForm(request.POST)
        
        if form.is_valid():
            # Extract cleaned data
            name = form.cleaned_data['name']
            email = form.cleaned_data['email']
            subject = form.cleaned_data['subject']
            message = form.cleaned_data['message']
            copy_to_self = form.cleaned_data['copy_to_self']
            
            # Here you would typically:
            # - Send an email
            # - Save to database
            # - Log the inquiry
            
            # Add success message
            messages.success(request, 'Thank you for your message!')
            
            return redirect('contact')
    else:
        form = ContactForm()
    
    return render(request, 'catalog/contact.html', {'form': form})

In [ ]:
# templates/catalog/contact.html

template_complete = """
{% extends 'base.html' %}

{% block title %}Contact Us{% endblock %}

{% block content %}
<h1>Contact Us</h1>

{% if messages %}
    {% for message in messages %}
        <div class="alert alert-{{ message.tags }}">{{ message }}</div>
    {% endfor %}
{% endif %}

<form method="post" novalidate>
    {% csrf_token %}
    
    {% if form.non_field_errors %}
        <div class="errors">
            {{ form.non_field_errors }}
        </div>
    {% endif %}
    
    {% for field in form %}
        <div class="form-group">
            {{ field.label_tag }}
            {{ field }}
            {% if field.help_text %}
                <small class="help-text">{{ field.help_text }}</small>
            {% endif %}
            {% if field.errors %}
                <span class="error">{{ field.errors.0 }}</span>
            {% endif %}
        </div>
    {% endfor %}
    
    <button type="submit">Send Message</button>
</form>
{% endblock %}
"""
print(template_complete)

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Form class** | Inherit from `forms.Form`, define fields as class attributes |
| **Field types** | `CharField`, `EmailField`, `IntegerField`, `ChoiceField`, etc. |
| **Rendering** | `{{ form.as_p }}` for quick, manual for control |
| **CSRF token** | Always include `{% csrf_token %}` in POST forms |
| **View pattern** | Check method → bind data → validate → process or re-display |
| **is_valid()** | Returns `True` if all fields pass validation |
| **cleaned_data** | Dictionary of validated, type-converted data |
| **Validators** | Built-in or custom functions to validate field values |
| **PRG pattern** | After success, redirect to prevent duplicate submissions |

<br>

### 🧠 EXERCISE 🧠, Create a Contact Form

---

Create a complete contact form for your bookstore project:

1. Create `catalog/forms.py` with a `ContactForm` class containing:
   - `name` (CharField, max 100 characters)
   - `email` (EmailField)
   - `phone` (CharField, optional)
   - `message` (CharField with Textarea widget, min 20 characters)

2. Create a view in `catalog/views.py` that:
   - Displays the form on GET
   - Validates and processes on POST
   - Prints the submitted data to console
   - Redirects back to the form after success

3. Create the template `templates/catalog/contact.html`

4. Add URL pattern in `catalog/urls.py`

5. Test by submitting the form with valid and invalid data

<details>
    <summary>▶️ Solution</summary>
    
**1. catalog/forms.py:**

```python
from django import forms


class ContactForm(forms.Form):
    """Contact form for customer inquiries."""
    
    name = forms.CharField(
        max_length=100,
        label='Your Name'
    )
    
    email = forms.EmailField(
        label='Email Address'
    )
    
    phone = forms.CharField(
        max_length=20,
        required=False,
        label='Phone Number'
    )
    
    message = forms.CharField(
        widget=forms.Textarea(attrs={'rows': 5}),
        min_length=20,
        label='Message',
        help_text='Please provide at least 20 characters.'
    )
```

**2. catalog/views.py:**

```python
from django.shortcuts import render, redirect
from .forms import ContactForm


def contact(request):
    if request.method == 'POST':
        form = ContactForm(request.POST)
        
        if form.is_valid():
            print("Form submitted:")
            print(f"  Name: {form.cleaned_data['name']}")
            print(f"  Email: {form.cleaned_data['email']}")
            print(f"  Phone: {form.cleaned_data['phone']}")
            print(f"  Message: {form.cleaned_data['message']}")
            
            return redirect('contact')
    else:
        form = ContactForm()
    
    return render(request, 'catalog/contact.html', {'form': form})
```

**3. templates/catalog/contact.html:**

```html
<!DOCTYPE html>
<html>
<head>
    <title>Contact Us - Bookstore</title>
</head>
<body>
    <h1>Contact Us</h1>
    
    <form method="post">
        {% csrf_token %}
        {{ form.as_p }}
        <button type="submit">Send Message</button>
    </form>
</body>
</html>
```

**4. catalog/urls.py:**

```python
from django.urls import path
from . import views

app_name = 'catalog'

urlpatterns = [
    path('contact/', views.contact, name='contact'),
]
```

**5. Test:**
- Run `python manage.py runserver`
- Visit `http://127.0.0.1:8000/contact/`
- Submit with missing required fields → see validation errors
- Submit with valid data → check console for printed output
</details>

<br>

### 🧠 EXERCISE 🧠, Build a Newsletter Signup

---

Create a newsletter signup form:

1. Create a `NewsletterForm` with:
   - `email` (EmailField, required)
   - `name` (CharField, optional)
   - `frequency` (ChoiceField with options: 'daily', 'weekly', 'monthly')
   - `interests` (MultipleChoiceField with checkboxes for: 'fiction', 'non-fiction', 'science', 'history')

2. Create a view that displays the form and processes submissions

3. When valid, print the selected interests to the console

**Hint:** For multiple checkboxes, use `forms.MultipleChoiceField` with `widget=forms.CheckboxSelectMultiple`

<details>
    <summary>▶️ Solution</summary>
    
**catalog/forms.py:**

```python
from django import forms


class NewsletterForm(forms.Form):
    """Newsletter signup form."""
    
    FREQUENCY_CHOICES = [
        ('daily', 'Daily'),
        ('weekly', 'Weekly'),
        ('monthly', 'Monthly'),
    ]
    
    INTEREST_CHOICES = [
        ('fiction', 'Fiction'),
        ('non-fiction', 'Non-Fiction'),
        ('science', 'Science'),
        ('history', 'History'),
    ]
    
    email = forms.EmailField(
        label='Email Address'
    )
    
    name = forms.CharField(
        max_length=100,
        required=False,
        label='Your Name (optional)'
    )
    
    frequency = forms.ChoiceField(
        choices=FREQUENCY_CHOICES,
        initial='weekly',
        label='How often?'
    )
    
    interests = forms.MultipleChoiceField(
        choices=INTEREST_CHOICES,
        widget=forms.CheckboxSelectMultiple,
        label='Topics of Interest'
    )
```

**catalog/views.py:**

```python
from django.shortcuts import render, redirect
from .forms import NewsletterForm


def newsletter_signup(request):
    if request.method == 'POST':
        form = NewsletterForm(request.POST)
        
        if form.is_valid():
            email = form.cleaned_data['email']
            interests = form.cleaned_data['interests']  # This is a list!
            
            print(f"New subscription: {email}")
            print(f"Interests: {', '.join(interests)}")
            
            return redirect('newsletter_signup')
    else:
        form = NewsletterForm()
    
    return render(request, 'catalog/newsletter.html', {'form': form})
```

**templates/catalog/newsletter.html:**

```html
<!DOCTYPE html>
<html>
<head>
    <title>Newsletter Signup</title>
</head>
<body>
    <h1>Subscribe to Our Newsletter</h1>
    
    <form method="post">
        {% csrf_token %}
        {{ form.as_p }}
        <button type="submit">Subscribe</button>
    </form>
</body>
</html>
```

**catalog/urls.py:**

```python
urlpatterns = [
    path('newsletter/', views.newsletter_signup, name='newsletter_signup'),
]
```
</details>

---